# InstructionAdherenceEvaluator usage

## 1. What this metric measures

Instruction Adherence asks whether explicit output constraints were followed (`instructions → output`). It requires `instructions + output`; optional `context` is supporting evidence and does not create new instructions. Optional descriptive `input` helps identify the task in traces and reports but is not sent to this metric's judge prompt.

## 2. Imports

In [ ]:
from idp_eval import (
    EvaluationCase,
    EvaluationFramework,
    InstructionAdherenceEvaluator,
    create_azure_judge,
)
from idp_eval.judges import AzureJudgeConfig

## 3. Judge configuration

Applications should inject configuration from their own settings/secrets layer. These are placeholders only. `create_gateway_judge(config=...)` is the equivalent gateway path; evaluators are backend-independent.

In [ ]:
azure_config = AzureJudgeConfig(
    model="your-azure-deployment",
    azure_endpoint="https://your-resource.openai.azure.com",
    tenant_id="your-tenant-id",
    client_id="your-client-id",
    client_secret="your-client-secret",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)
judge = create_azure_judge(config=azure_config)

## 4. Basic single-output example

A plain multiline string is the common production instruction shape. Dict, list, and nested structured instructions are also supported through `render_value()`. Context below is evidence only for the explicit approved-options instruction.

In [ ]:
instructions = """
Generate exactly 3 options.
Every option must contain a title.
Do not include implementation details.
Only use approved options from the supplied context.
"""

approved_context = {"approved_options": ["Option A", "Option B", "Option C"]}
options = [
    {"title": "Option A"},
    {"title": "Option B"},
    {"title": "Option C"},
]
case = EvaluationCase(
    case_id="example-001",
    input="Generate approved deployment options.",
    instructions=instructions,
    context=approved_context,
    output=options,
)
evaluator = InstructionAdherenceEvaluator(judge, verbose=True)
framework = EvaluationFramework(judge=judge, evaluators=[evaluator])
result = framework.evaluate(case)["instruction_adherence"]

## 5. Understanding the result

The judge identifies distinct checkable instructions and labels each `followed` or `violated`. Python maps those statuses to `1.0` or `0.0` and computes their mean. With `verbose=True`, `details["instructions"]` is the complete audit trail.

In [ ]:
{
    "score": result.score,
    "label": result.label,
    "explanation": result.explanation,
    "details": result.details,
}
result.details["instructions"]

## 6. Multiple outputs and `evaluation_scope`

The default `combined` scope judges the collection as one output, which is the natural choice for collection-level rules such as “exactly 3 options” and “every option has a title.” `individual` applies the same full instruction set to each item independently, so collection-level constraints may intentionally fail per item. `both` runs the combined view and every item as separate logical evaluations and root traces.

In [ ]:
combined_case = EvaluationCase(
    case_id="example-001", input="Generate approved options.",
    instructions=instructions, context=approved_context, output=options,
    evaluation_scope="combined",
)
individual_case = EvaluationCase(
    case_id="example-001", input="Generate approved options.",
    instructions=instructions, context=approved_context, output=options,
    evaluation_scope="individual",
)
both_case = EvaluationCase(
    case_id="example-001", input="Generate approved options.",
    instructions=instructions, context=approved_context, output=options,
    evaluation_scope="both",
)
combined_results = framework.evaluate(combined_case)
individual_results = framework.evaluate(individual_case)
both_results = framework.evaluate(both_case)

combined_results["instruction_adherence"]
individual_results["individual"][0]["instruction_adherence"]
both_results["combined"]["instruction_adherence"]

Return shapes: combined returns `{'instruction_adherence': EvaluationResult(...)}`; individual returns `{'combined': None, 'individual': [{'instruction_adherence': ...}, ...]}`; both returns the combined mapping plus the individual list. Child case IDs are `example-001:0`, `example-001:1`, and `example-001:2`.

## 7. Optional async usage

Jupyter supports top-level `await`, so no event-loop wrapper is needed. The framework enforces the shared judge-call concurrency limit.

In [ ]:
async_result = await framework.a_evaluate(case, max_concurrency=4)
async_result["instruction_adherence"]

## 8. `evaluate_many()`

Bulk evaluation accepts unrelated cases, preserves input order, and lets every case choose its own scope. The async variant enforces one shared concurrency limit across all judge calls.

In [ ]:
case_a = EvaluationCase(input="Generate approved options.", instructions=instructions, context=approved_context, output=options)
case_b = EvaluationCase(input="Review each option.", instructions="Every option must contain a title.", output=options, evaluation_scope="individual")
many_results = framework.evaluate_many([case_a, case_b])
async_many_results = await framework.a_evaluate_many([case_a, case_b], max_concurrency=4)

## 9. Close resources

In [ ]:
judge.close()